# Lesson 02 - 探索 Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** 是一個用於構建 AI 代理的統一框架。它提供了一個乾淨且可組合的架構，包含四個核心構建塊：

- **Client** – 連接到 AI 模型端點並處理通信
- **Agent** – 將指令和工具定義包裝到客戶端中
- **Tools** – 通過模型可調用的自定義函數擴展代理能力
- **Session** – 維護多輪交互的對話歷史

在本課中，我們將使用這些概念構建一個 **旅遊預訂代理**，用來檢查目的地的可用性。


## 設定


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## 了解代理框架架構

Microsoft 代理框架遵循分層架構：

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – 一個 `AzureAIProjectAgentProvider` 連接到 Azure OpenAI 部署。它負責身份驗證、請求格式化和回應解析。
2. **Agent** – 由 client 通過 `provider.create_agent()` 創建，代理結合模型存取、指令（系統提示）與工具。
3. **Tools** – 由 `@tool` 裝飾的 Python 函數，代理可以呼叫它們來執行動作或擷取資料。
4. **Session** – 一個 `AgentSession` 物件（通過 `agent.create_session()` 創建），用來儲存對話歷史，使多輪對話中代理能記住先前的上下文。

讓我們逐步構建每個層次。


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 使用 @tool 裝飾器新增工具

工具讓代理執行超出產生文字之外的動作。`@tool` 裝飾器會把一般的 Python 函式轉換成代理可以呼叫的功能。

重點：
- 使用 `Annotated[type, "description"]` 讓模型了解每個參數。
- 文件字串成為模型看到的工具說明。
- `approval_mode="never_require"` 表示工具會自動執行，無需使用者確認。


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## 使用工具建立代理程式

現在我們將用戶端、指令和工具組合成一個代理程式。`instructions` 作為系統提示 — 它們定義代理程式的人格與行為。


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## 多輪對話及會話管理

一個 `AgentSession`（透過 `agent.create_session()` 建立）會追踪對話中的所有訊息。通過在每次調用 `agent.run()` 時傳入相同的會話，代理程式可以存取完整的對話歷史並回顧早前的訊息。

我們傳入 `tools=[check_destination_availability]`，以便代理程式在每一輪都能調用我們的可用性檢查器。


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Summary

在本課程中，您探討了 Microsoft Agent Framework 的四大支柱：

| Concept | What You Learned |
|---------|------------------|
| **Client** | `AzureAIProjectAgentProvider` 使用基於憑證的身份驗證連接至 Azure OpenAI |
| **Agent** | `provider.create_agent()` 將模型連接與指令及名稱打包 |
| **Tools** | `@tool` 裝飾器將 Python 函數公開給代理調用 |
| **Session** | `agent.create_session()` 維持多輪對話的歷史記錄 |

這些構建模組共同組成代理，能進行自然對話、調用外部函數並維持上下文——為後續課程中更高級的代理模式奠定基礎。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：  
本文件乃使用人工智能翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 所翻譯。雖然我們致力於追求準確性，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議採用專業人工翻譯。我們不對因使用此翻譯而引起的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
